# Feature Engineering — E-Commerce Fraud Data

**Objective:** Engineer behavioral, temporal, and geolocation features from `Fraud_Data.csv`.

Features created:
- `hour_of_day` — purchase hour (0–23)
- `day_of_week` — purchase day (0=Mon, 6=Sun)
- `time_since_signup` — hours between signup and purchase
- `tx_count_24h` — transaction velocity (last 24h per user)
- `device_user_count` — how many users share the same device
- `country` — IP geolocation

Followed by: normalization, one-hot encoding, and SMOTE resampling.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_fraud_data, load_ip_country
from src.preprocessing import (
    drop_duplicates, handle_missing, merge_ip_country,
    scale_features, encode_categoricals
)
from src.feature_engineering import engineer_all_features
from src.resampling import get_class_distribution, apply_smote

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Setup complete')

## 1. Load and Clean

In [ ]:
fraud_df = load_fraud_data()
ip_df = load_ip_country()

fraud_df = drop_duplicates(fraud_df)
fraud_df = handle_missing(fraud_df, strategy='drop')
fraud_df = merge_ip_country(fraud_df, ip_df)
print(f'Shape after cleaning: {fraud_df.shape}')

## 2. Feature Engineering

In [ ]:
fraud_df = engineer_all_features(fraud_df, window_hours=24)
print(f'Shape after feature engineering: {fraud_df.shape}')
fraud_df[['time_since_signup', 'tx_count_24h', 'device_user_count',
           'hour_of_day', 'day_of_week']].describe()

In [ ]:
# Visualize key engineered features vs fraud
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.histplot(data=fraud_df, x='time_since_signup', hue='class',
             bins=60, ax=axes[0][0], log_scale=(True, False),
             palette={0:'steelblue', 1:'tomato'})
axes[0][0].set_title('Time Since Signup (hours) — log scale')

sns.histplot(data=fraud_df, x='hour_of_day', hue='class',
             bins=24, ax=axes[0][1],
             palette={0:'steelblue', 1:'tomato'})
axes[0][1].set_title('Hour of Day')

sns.histplot(data=fraud_df, x='tx_count_24h', hue='class',
             bins=20, ax=axes[1][0],
             palette={0:'steelblue', 1:'tomato'})
axes[1][0].set_title('Transaction Count (24h window)')

sns.countplot(data=fraud_df, x='day_of_week', hue='class',
              ax=axes[1][1], palette={0:'steelblue', 1:'tomato'})
axes[1][1].set_title('Day of Week')
axes[1][1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])

plt.tight_layout()
plt.show()

## 3. Data Transformation

In [ ]:
# Drop columns not useful for modeling
drop_cols = ['user_id', 'device_id', 'signup_time', 'purchase_time',
             'ip_address', 'ip_int']
fraud_df = fraud_df.drop(columns=[c for c in drop_cols if c in fraud_df.columns])

# One-hot encode categorical features
cat_cols = ['source', 'browser', 'sex', 'country']
fraud_df = encode_categoricals(fraud_df, cols=cat_cols)
print(f'Shape after encoding: {fraud_df.shape}')

In [ ]:
# Separate features and target
X = fraud_df.drop(columns=['class'])
y = fraud_df['class']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

# Scale numerical features
num_cols = ['purchase_value', 'age', 'time_since_signup',
            'tx_count_24h', 'device_user_count', 'hour_of_day', 'day_of_week']
num_cols = [c for c in num_cols if c in X_train.columns]

X_train, scaler = scale_features(X_train, num_cols, method='standard')
X_test[num_cols] = scaler.transform(X_test[num_cols])

## 4. Handle Class Imbalance with SMOTE

**Justification:** SMOTE (Synthetic Minority Over-sampling Technique) is chosen over simple undersampling because:
- It preserves all majority class information (undersampling discards data).
- It generates synthetic, realistic minority samples rather than duplicates.
- Applied **only on the training set** to prevent data leakage.

Alternative considered: RandomUnderSampler — acceptable for very large datasets where speed is critical.

In [ ]:
print('=== Before SMOTE ===')
get_class_distribution(y_train, label='Training Set')

X_train_res, y_train_res = apply_smote(X_train, y_train, random_state=42)

print('\n=== After SMOTE ===')
get_class_distribution(y_train_res, label='Training Set (resampled)')

In [ ]:
# Save processed datasets for modeling
import joblib

pd.DataFrame(X_train_res, columns=X_train.columns).to_csv(
    '../data/processed/fraud_X_train.csv', index=False)
pd.Series(y_train_res, name='class').to_csv(
    '../data/processed/fraud_y_train.csv', index=False)
X_test.to_csv('../data/processed/fraud_X_test.csv', index=False)
y_test.to_csv('../data/processed/fraud_y_test.csv', index=False)
joblib.dump(scaler, '../models/fraud_scaler.pkl')

print('All processed data saved to data/processed/')

## Summary

| Feature | Type | Signal Strength | Notes |
|---|---|---|---|
| `time_since_signup` | Temporal | ★★★ | Short gaps strongly predict fraud |
| `tx_count_24h` | Velocity | ★★★ | High burst = suspicious |
| `hour_of_day` | Temporal | ★★ | Late-night activity elevated |
| `device_user_count` | Behavioral | ★★ | Shared devices = risk |
| `country` | Geo | ★★ | Some countries show 3x avg fraud rate |
| `purchase_value` | Transactional | ★ | Modest predictive power alone |

**Next step:** Model training → `modeling.ipynb`